In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import os
from datetime import datetime
from scipy.signal import resample
from scipy.interpolate import interp1d


# --- helpers ---

def convert_str_to_datetime(datetime_str: str) -> datetime:
    return datetime.strptime(datetime_str, '%Y-%m-%d %H:%M:%S.%f')


def interpolate(t, x, sample_rate, axis=0, kind='linear', threshold=3):
    """
    Linearly interpolate time series to a uniform sample rate.
    """
    t = np.array([convert_str_to_datetime(ts).timestamp() for ts in t])
    t0, tN = t[0], t[-1]
    dt = 1 / sample_rate
    t_ = np.arange(t0, tN + dt / 2, dt).round(threshold)
    f = interp1d(t, x, axis=axis, kind=kind, copy=False, fill_value='extrapolate')
    return t_, f(t_)


def synchronize(left_wrist_t, acc, loc):
    """
    Resample to match left wrist timeline and interpolate to 50 Hz.
    """
    data_len = len(left_wrist_t)
    if loc != 'left wrist':
        acc = resample(acc, data_len)
    interp_time, interp_acc = interpolate(left_wrist_t.copy(), acc, sample_rate=50)
    return interp_time, interp_acc


# --- main function ---

def interpolate_sensor_data(data_path: Path, output_dir: Path):
    """
    Interpolates sensor data (50 Hz) for each subject.
    Skips subjects missing any of the 4 required sensor files.
    """
    subjects = sorted([s for s in os.listdir(data_path) if not s.startswith(".")])

    for subject in subjects:
        subject_path = data_path / subject
        output_subject_path = output_dir / subject
        output_subject_path.mkdir(parents=True, exist_ok=True)

        print(f"\n Checking subject: {subject}")

        # Required files
        required_files = [
            "left finger.csv",
            "right finger.csv",
            "left wrist.csv",
            "right wrist.csv"
        ]

        available_files = os.listdir(subject_path)
        missing_files = [f for f in required_files if f not in available_files]

        # Skip if any required file is missing
        if missing_files:
            print(f" Skipping {subject} — missing files: {missing_files}")
            continue

        try:
            # --- Read CSVs ---
            left_finger = pd.read_csv(subject_path / "left finger.csv")
            right_finger = pd.read_csv(subject_path / "right finger.csv")
            left_wrist = pd.read_csv(subject_path / "left wrist.csv")
            right_wrist = pd.read_csv(subject_path / "right wrist.csv")

            # --- Synchronize all to left wrist ---
            synchronized_left_finger_time, synchronized_left_finger_acc = synchronize(
                left_wrist_t=left_wrist["timestamp"],
                acc=left_finger["value"],
                loc="left finger"
            )

            interpolated_data = {
                "left finger": synchronized_left_finger_acc,
                "right finger": synchronize(left_wrist["timestamp"], right_finger["value"], "right finger")[1],
                "left wrist": synchronize(left_wrist["timestamp"], left_wrist["value"], "left wrist")[1],
                "right wrist": synchronize(left_wrist["timestamp"], right_wrist["value"], "right wrist")[1],
            }

            df_interpolated = pd.DataFrame({
                "timestamp": synchronized_left_finger_time,
                "left finger": interpolated_data["left finger"],
                "right finger": interpolated_data["right finger"],
                "left wrist": interpolated_data["left wrist"],
                "right wrist": interpolated_data["right wrist"]
            })

            output_file = output_subject_path / "interpolated.csv"
            df_interpolated.to_csv(output_file, index=False)
            print(f" Saved: {output_file}")

        except Exception as e:
            print(f" Error processing {subject}: {e}")
            continue


# -------------------------------
# Example usage
# -------------------------------
interpolate_sensor_data(
    data_path=Path("path/to/your/data"),
    output_dir=Path("path/to/your/data")
)
